In [1]:
import requests
import pandas as pd
import numpy as np
import polars as pl

In [2]:
# Fetch help contacts data
help_contacts = requests.get("http://api.justgood.work/api/v2/help-contacts")
help_contacts_json = help_contacts.json()["data"]

In [3]:
# Convert help contacts json to a pandas Dataframe
hc_df = pd.DataFrame.from_records(help_contacts_json)

# view the first 2 records
hc_df.head()

,_id,deviceId,appId,locale,route,phoneNumber,organization,email,createdAt,updatedAt,__v,language
0,6463a77057f25744aff36429,jgw-vanilla-afe14226d1d2386e-TJK -> GBR,jgw-vanilla,TJK,TJK -> GBR,08000 121 700,,,2023-05-16T15:55:28.363Z,2023-05-16T15:55:28.363Z,0,NaN
1,640ef2983f51d833637453f5,jgw-5158254C-33FD-4A66-81D3-1F9FD042AD26-madag...,jgw,madagascar,madagascar -> mauritius,None,In Mauritius: Ministry of Labour,mol@govmu.org,2023-03-13T09:53:28.125Z,2023-03-13T09:53:28.125Z,0,english
2,638add6d292f141036c92d08,jgw-undefined-bangladesh -> mauritius,jgw,bangladesh,bangladesh -> mauritius,+২৩০ ২০৭ ২৭১১,মরিশাসঃ শ্রম মন্ত্রণালয়,None,2022-12-03T05:23:57.694Z,2022-12-03T05:23:57.694Z,0,বাংলা
3,638add6d292f141036c92d07,jgw-undefined-bangladesh -> mauritius,jgw,bangladesh,bangladesh -> mauritius,+২৩০ ২০৭ ২৭১১,মরিশাসঃ শ্রম মন্ত্রণালয়,None,2022-12-03T05:23:57.494Z,2022-12-03T05:23:57.494Z,0,বাংলা
4,638add6d292f141036c92d06,jgw-undefined-bangladesh -> mauritius,jgw,bangladesh,bangladesh -> mauritius,+২৩০ ২০৭ ২৭১১,মরিশাসঃ শ্রম মন্ত্রণালয়,None,2022-12-03T05:23:57.275Z,2022-12-03T05:23:57.275Z,0,বাংলা


In [4]:
# generics
def count_unique(df, field):
    return df[field].nunique()

In [5]:
# phone_email_comparision

def phone_email_comparision(df):
    return {
        "phone": count_unique(df, "phoneNumber"),
        "email": count_unique(df, "email")
    }
    
phone_email_comparision(hc_df)

{'phone': 70, 'email': 11}

In [6]:
# phone numbers and emails by route    
def aggregate_by_group(df, result_fields, group_field):
    df[group_field] = df[group_field].replace("", "Not specified")
    
    aggregation_dict = {field: lambda values: values.count() for field in result_fields}
    grouped_data = df.groupby(group_field)[result_fields].agg(aggregation_dict)
    
    result = {
        "labels": grouped_data.index.values,
    }
    
    for field in result_fields:
        result[field] = grouped_data[field].values
    
    return result

dd = aggregate_by_group(hc_df, result_fields=["phoneNumber", "email"],group_field="route")
dd

{'labels': array(['Bangladesh -> uk', 'Bulgaria -> uk', 'Greece -> uk',
        'India -> uk', 'Italy -> uk',
        'Other - The rest of the world -> uk', 'Romania -> uk',
        'TJK -> GBR', 'Ukraine -> uk', 'bangladesh -> malaysia',
        'bangladesh -> mauritius', 'burma -> malaysia',
        'india -> malaysia', 'indonesia -> malaysia', 'kenya -> qatar',
        'kenya -> saudi', 'kenya -> uae', 'madagascar -> mauritius',
        'nepal -> malaysia', 'pakistan -> malaysia', 'uganda -> saudi',
        'vietnam -> malaysia'], dtype=object),
 'phoneNumber': array([ 2,  3, 35,  1,  1,  1, 12,  1,  2, 59, 14, 32,  5, 35, 10,  3,  1,
         7, 25,  0,  3,  2]),
 'email': array([2, 0, 0, 0, 0, 1, 0, 1, 1, 9, 4, 4, 0, 4, 1, 0, 3, 4, 2, 1, 0, 0])}

In [15]:
def create_label_dictionary(labels, values):
    new_dict = {}

    for label, value in zip(labels, values):
        new_dict[label] = value

    return new_dict

phone_dict = create_label_dictionary(dd.get("labels"), dd.get("phoneNumber"))
email_dict = create_label_dictionary(dd.get("labels"), dd.get("email"))
phone_email_dict = {**phone_dict, **email_dict}
phone_email_dict

{'Bangladesh -> uk': 2,
 'Bulgaria -> uk': 0,
 'Greece -> uk': 0,
 'India -> uk': 0,
 'Italy -> uk': 0,
 'Other - The rest of the world -> uk': 1,
 'Romania -> uk': 0,
 'TJK -> GBR': 1,
 'Ukraine -> uk': 1,
 'bangladesh -> malaysia': 9,
 'bangladesh -> mauritius': 4,
 'burma -> malaysia': 4,
 'india -> malaysia': 0,
 'indonesia -> malaysia': 4,
 'kenya -> qatar': 1,
 'kenya -> saudi': 0,
 'kenya -> uae': 3,
 'madagascar -> mauritius': 4,
 'nepal -> malaysia': 2,
 'pakistan -> malaysia': 1,
 'uganda -> saudi': 0,
 'vietnam -> malaysia': 0}